# Whippy_TTS — Colab End-to-End Test

This notebook runs the full integration pipeline:

```
user text → Whippy Chat API → Chatterbox TTS → output.wav → playable audio
```

**Before you start**

1. Set the runtime to a **GPU** instance: **Runtime → Change runtime type → T4 GPU** (or better).
2. Start your local Whippy API and expose it with a temporary Cloudflare Tunnel.
3. Add Colab Secrets (🔑 in the left sidebar) or be ready to enter credentials manually.
4. Have a clean `reference.wav` clip ready to upload (8–15 seconds; see `voices/README.md`).

This notebook reuses the repository modules:

- `app/whippy_client.py`
- `app/config.py`
- `config/voice_config.json`
- `scripts/test_whippy_to_speech.py` (same flow, executed step-by-step below)

No secrets are stored in this notebook file.

## GPU verification

Chatterbox speech generation requires a CUDA GPU. This cell fails fast if the runtime is CPU-only.

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA is not available. Switch to Runtime → Change runtime type → GPU, then rerun."
    )

print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))
print("CUDA version:", torch.version.cuda)

## Clone repository

Clone the Whippy_TTS repository and move into it. Push your branch to GitHub first if you need the latest local changes.

In [ ]:
import os
from pathlib import Path

REPO_URL = "https://github.com/Ritwik7631/Whippy_TTS.git"
REPO_DIR = Path("/content/Whippy_TTS")
BRANCH = "feature/whippy-integration"

if REPO_DIR.exists():
    print(f"Repository already exists at {REPO_DIR}")
else:
    !git clone {REPO_URL} {REPO_DIR}

%cd {REPO_DIR}
!git checkout {BRANCH}

print("Working directory:", Path.cwd())

## Install dependencies

Install project requirements. Colab already includes a CUDA-enabled PyTorch build, so we do **not** reinstall PyTorch.

In [ ]:
from pathlib import Path

requirements_path = Path("requirements.txt")
if not requirements_path.exists():
    raise FileNotFoundError(f"Missing {requirements_path.resolve()}")

!pip install -q -r requirements.txt
print("Dependencies installed from requirements.txt")

## Verify Python version

Confirm the Colab Python version and that required project files exist.

In [ ]:
import sys
from pathlib import Path

REPO_DIR = Path("/content/Whippy_TTS")

if not REPO_DIR.exists():
    raise FileNotFoundError(
        "Repository not found at /content/Whippy_TTS. Run the Clone repository cell first."
    )

%cd /content/Whippy_TTS

print("Python version:", sys.version)
print("Working directory:", Path.cwd())

print("\nGit branch:")
!git rev-parse --abbrev-ref HEAD

print("\nLatest commit:")
!git log -1 --oneline

required_paths = [
    REPO_DIR / "app/whippy_client.py",
    REPO_DIR / "app/config.py",
    REPO_DIR / "config/voice_config.json",
    REPO_DIR / "scripts/test_whippy_to_speech.py",
    REPO_DIR / "requirements.txt",
    REPO_DIR / ".env.example",
]

missing = [path for path in required_paths if not path.exists()]
if missing:
    missing_lines = "\n".join(f"  - {path.relative_to(REPO_DIR)}" for path in missing)
    raise FileNotFoundError(
        "Missing required repository files:\n"
        f"{missing_lines}\n\n"
        "Colab clones from GitHub, not your local PC. "
        "On your machine, commit and push these files to "
        "feature/whippy-integration, then rerun the Clone repository cell:\n\n"
        "  git add app/whippy_client.py scripts/ .env.example\n"
        "  git commit -m \"Add Whippy chat client and Colab test scripts\"\n"
        "  git push -u origin feature/whippy-integration"
    )

print("All required project files are present.")

## Configure environment variables

Whippy credentials are loaded from Colab Secrets when available.

Add these secrets in the Colab sidebar (**🔑 Secrets**):

| Secret | Description |
| --- | --- |
| `WHIPPY_BASE_URL` | Public tunnel URL, e.g. `https://your-subdomain.trycloudflare.com` |
| `WHIPPY_API_KEY` | Pow session access token |
| `WHIPPY_AGENT_ID` | Agent UUID |
| `WHIPPY_ORGANIZATION_ID` | Organization UUID |

If a secret is missing, this cell prompts you to enter the value manually.

In [ ]:
import os
from getpass import getpass
from pathlib import Path

SECRET_NAMES = [
    "WHIPPY_BASE_URL",
    "WHIPPY_API_KEY",
    "WHIPPY_AGENT_ID",
    "WHIPPY_ORGANIZATION_ID",
]


def load_secret(name: str) -> str:
    try:
        from google.colab import userdata

        value = userdata.get(name).strip()
        if value:
            print(f"Loaded {name} from Colab Secrets")
            return value
    except Exception:
        pass

    prompt = f"Enter {name}: "
    if name == "WHIPPY_API_KEY":
        value = getpass(prompt)
    else:
        value = input(prompt)

    value = value.strip()
    if not value:
        raise ValueError(f"{name} is required")
    return value


env_values = {name: load_secret(name) for name in SECRET_NAMES}
env_values["WHIPPY_BASE_URL"] = env_values["WHIPPY_BASE_URL"].rstrip("/")

for name, value in env_values.items():
    os.environ[name] = value

env_lines = [f"{name}={value}" for name, value in env_values.items()]
Path(".env").write_text("\n".join(env_lines) + "\n", encoding="utf-8")

print("Wrote .env with", len(env_lines), "variables.")
print("WHIPPY_BASE_URL:", env_values["WHIPPY_BASE_URL"])

## Upload `reference.wav`

Upload your voice reference clip. The file is moved automatically to `voices/reference.wav`, which is the path defined in `config/voice_config.json`.

In [ ]:
from pathlib import Path
from google.colab import files

voices_dir = Path("voices")
voices_dir.mkdir(parents=True, exist_ok=True)
target_path = voices_dir / "reference.wav"

print("Upload a WAV file named reference.wav")
uploaded = files.upload()

if "reference.wav" not in uploaded:
    raise RuntimeError(
        "Expected an uploaded file named reference.wav. "
        "Rename your clip or re-upload with that exact filename."
    )

target_path.write_bytes(uploaded["reference.wav"])
print(f"Saved reference audio to {target_path.resolve()}")

## Verify CUDA

Double-check CUDA after installing dependencies.

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError("CUDA is required for Chatterbox generation.")

print("CUDA ready on:", torch.cuda.get_device_name(0))

## Load Chatterbox

Load the Chatterbox model using the same library and device settings as `scripts/test_whippy_to_speech.py`.

In [ ]:
import sys
from pathlib import Path

import torch
from chatterbox.tts import ChatterboxTTS

ROOT_DIR = Path.cwd()
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

from app.config import load_voice_config, resolve_repo_path

voice_config = load_voice_config()
reference_audio = resolve_repo_path(voice_config["reference_audio_path"])
generation_config = voice_config["generation"]
output_audio = resolve_repo_path("outputs/output.wav")

if not reference_audio.exists():
    raise FileNotFoundError(f"Missing reference audio: {reference_audio}")

print("Loading Chatterbox model...")
chatterbox_model = ChatterboxTTS.from_pretrained(device="cuda")
print("Chatterbox model loaded.")
print("Reference audio:", reference_audio.resolve())

## Call the Whippy Chat API

Send the same test message used by `scripts/test_whippy_to_speech.py` through `app/whippy_client.py`.

In [ ]:
import os
import sys
from pathlib import Path

from dotenv import load_dotenv

ROOT_DIR = Path.cwd()
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

from app.whippy_client import WhippyApiError, WhippyClient, WhippyConfig

TEST_MESSAGE = "Hello"

load_dotenv(ROOT_DIR / ".env")
whippy_config = WhippyConfig.from_env(os.environ)
client = WhippyClient(whippy_config)

print("Whippy base URL:", whippy_config.base_url)
print("Sending test message:", TEST_MESSAGE)

try:
    chat_data = client.chat([{"role": "user", "content": TEST_MESSAGE}])
except WhippyApiError as error:
    raise RuntimeError(f"Whippy API request failed: {error}") from error

response_text = chat_data.get("response")
if not isinstance(response_text, str) or not response_text.strip():
    raise RuntimeError("Whippy response is missing data.response text")

print("\nWhippy agent response:\n")
print(response_text)

## Generate speech

Convert the Whippy agent response text into speech with Chatterbox, using settings from `config/voice_config.json`.

In [ ]:
import time

print("Generating speech from Whippy response...")
generation_started_at = time.perf_counter()

waveform = chatterbox_model.generate(
    response_text,
    audio_prompt_path=str(reference_audio),
    exaggeration=generation_config["exaggeration"],
    cfg_weight=generation_config["cfg_weight"],
)

generation_seconds = time.perf_counter() - generation_started_at
print(f"Generation time: {generation_seconds:.2f}s")

## Save `output.wav`

Write the generated audio to `outputs/output.wav`, matching `scripts/test_whippy_to_speech.py`.

In [ ]:
import torchaudio as ta

output_audio.parent.mkdir(parents=True, exist_ok=True)

ta.save(
    str(output_audio),
    waveform.detach().cpu(),
    chatterbox_model.sr,
)

print("Saved output audio to:", output_audio.resolve())

## Play `output.wav`

Review the results and listen to the generated audio.

In [ ]:
from IPython.display import Audio, display
from pathlib import Path

if not output_audio.exists():
    raise FileNotFoundError(output_audio)

print("Whippy agent response:")
print(response_text)
print()
print(f"TTS generation time: {generation_seconds:.2f}s")
print(f"Output path: {output_audio.resolve()}")
print()
print("Playback:")
display(Audio(filename=str(output_audio), autoplay=False))

## Optional: run the packaged script

The cells above mirror `scripts/test_whippy_to_speech.py` step-by-step. To run the single packaged command instead:

```python
!python scripts/test_whippy_to_speech.py
```